 # Finetune a GET Model on PBMC 10k Multiome (Leaving out chr10, chr11, no cell type leave out)


 This tutorial demonstrates how to train a GET model to predict expression in ATAC-seq peaks using motif information. We'll cover:

 1. Loading and configuring the model

 2. Finetune from a pretrained expression prediction GET model

 3. Perform various analysis using `gcell` package



 ## Setup

 First, let's import the necessary modules and set up our configuration.
 
 Note:
 If you run from a Mac, make sure you use the jupyter notebook rather than the VSCode interactive python editor as the later seems to have issue with multiple workers.
 If you run from Linux, both should work fine.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from gcell.cell.celltype import GETHydraCellType

from get_model.config.config import load_config
from get_model.run_region import run_zarr as run


 ## Finetune



 We'll start by loading a predefined configuration and customizing it for our needs.

 The base configuration is in `get_model/config/finetune_tutorial_pbmc.yaml`

In [ ]:
if not Path('./checkpoint-best.pth').exists():
    s3_checkpoint_url = "s3://2023-get-xf2217/get_demo/checkpoints/regulatory_inference_checkpoint_fetal_adult/finetune_fetal_adult_leaveout_astrocyte/checkpoint-best.pth"
    ! aws s3 cp $s3_checkpoint_url ./checkpoint-best.pth --no-sign-request

> Note: In the paper, we mainly used binary ATAC signal trained model for motif interpretation analysis. As it's hard to say whether there are mutual causal relationship between transcription and accessibility. If accessibility is added to the model, potentially it will absorb some TF's effect to itself, thereby making the interpretation more difficult. However, if the goal is to represent the cell states as precisely as possible and use the model for other downstream tasks (e.g. enhancer target prediction), adding the accessibility signal is probably better.

In [ ]:
celltype_for_modeling = [
    'memory_b',
    'cd14_mono',
    'gdt',
    'cd8_tem_1',
    'naive_b',
    'mait',
    'intermediate_b',
    'cd4_naive',
    'cd8_tem_2',
    'cd8_naive',
    'cd4_tem',
    'cd4_tcm',
    'cd16_mono',
    'nk',
    'cdc',
    'treg']
cfg = load_config('finetune_tutorial_pbmc') # load the predefined finetune tutorial config
cfg.stage = 'fit'
cfg.run.run_name = 'training_from_finetune_lora_chr10_chr11_split_QATAC'
cfg.dataset.quantitative_atac = True # We use binary ATAC signal for motif interpretation analysis
cfg.dataset.zarr_path = "./pbmc10k_multiome.zarr" # set the path to the data zarr file we just processed
cfg.dataset.celltypes = ','.join(celltype_for_modeling) # the celltypes you want to finetune
cfg.finetune.checkpoint = "./checkpoint-best.pth" # set the path to the pretrained checkpoint we want to finetune from
cfg.dataset.leave_out_celltypes = '' # set the celltypes you want to leave out, '' here means no celltype leave out
cfg.dataset.leave_out_chromosomes = 'chr10,chr11' # set the chromosomes you want to leave out, '' here means no chromosome leave out
cfg.machine.num_devices=1 # use 0 for cpu training; >=1 for gpu training
cfg.machine.batch_size=8 # batch size for training; check `nvidia-smi` to see the available GPU memory
print(f"output path: {cfg.machine.output_dir}/{cfg.run.project_name}/{cfg.run.run_name}")
print(f"training for {cfg.training.epochs} epochs")

Now we can start the finetuning

In [ ]:
trainer = run(cfg) # run the finetuning, takes around 2 hours on one RTX 3090
print("checkpoint path:", trainer.checkpoint_callback.best_model_path)

It's almost converged so I stopped it.

# Interpretation

After finetuning, we can use the checkpoint to predict expression of all accessible genes and generate jacobian matrix of (peak x motif) for every predicted genes. 
To start, we need to collect the checkpoint we produced and switch to `predict` stage. Here, let's focus on CD4 Naive cell and we need to set `cfg.leave_out_celltypes` to `cd4_naive` for the model to predict gene expression in this cell type.

In [ ]:
use_checkpoint = '/home/xf2217/output/finetune_pbmc10k_multiome/training_from_finetune_lora_chr10_chr11_split_QATAC/checkpoints/best.ckpt'
cfg.stage = 'predict'
cfg.finetune.resume_ckpt = use_checkpoint
# in predict stage, leave out the celltype you want to interpret
cfg.run.use_wandb=False # disable wandb logging when predicting
cfg.task.layer_names = [] # set to empty list to disable intermediate layer interpretation
cfg.task.gene_list = None # set to None to predict all genes; otherwise you can specify a list of genes as 'MYC,SOX10,SOX2,RET', only genes with promoter open will be used
# loop through all celltypes and run the predict stage
cfg.run.run_name='interpret_training_from_finetune_lora_chr10_chr11_split_QATAC'

for celltype in celltype_for_modeling:
    cfg.dataset.leave_out_celltypes = celltype 
    trainer = run(cfg)

# Pearson 0.896
# R^2 0.793
# Spearman 0.803



As you can see, the results is now saved to `finetune_pbmc10k_multiome/interpret_training_from_finetune_lora_cd4_tcm_no_chr_split/cd4_naive.zarr`. Now we can use the `GETHydraCellType` class from `gcell` to load it.

## Load interpretation result as `GETHydraCellType`

In [ ]:
# get the celltype object based on the configuration
celltype = 'cd4_naive'
cfg = load_config('finetune_tutorial_pbmc')
cfg.run.run_name='interpret_training_from_finetune_lora_chr10_chr11_split_QATAC'
cfg.dataset.leave_out_celltypes = celltype
gene_annot_dict = {}
# cd14_mono.zarr  cd4_naive.zarr  cd8_naive.zarr  cd8_tem_1.zarr  cd8_tem_2.zarr  csv_logs  gdt.zarr  intermediate_b.zarr  mait.zarr  memory_b.zarr  naive_b.zarr
for celltype in ['cd14_mono',  'cd8_tem_1', 'cd8_tem_2', 'gdt', 'intermediate_b', 'mait', 'memory_b', 'naive_b']:
    try:
        cfg.dataset.leave_out_celltypes = celltype
        hydra_celltype = GETHydraCellType.from_config(cfg, celltype=celltype) # if you ran the predict stage multiple times on different celltypes, you can specify the celltype you want to load here
        gene_annot_dict[celltype] = hydra_celltype.gene_annot
    except Exception as e:
        print(f"Error loading {celltype}: {e}")
        continue


In [ ]:
# for each gene, collect the predicted and observed expression from all celltypes
# get a common intersected gene list first
import numpy as np
import pandas as pd

common_gene_list = set(gene_annot_dict[celltype_for_modeling[3]].index)
for celltype in celltype_for_modeling:
    try:
        common_gene_list = common_gene_list.intersection(set(gene_annot_dict[celltype].index))
    except Exception as e:
        print(f"Error loading {celltype}: {e}")
        continue
# then collect the expression
for celltype in celltype_for_modeling:
    try:
        gene_annot_dict[celltype] = gene_annot_dict[celltype].loc[np.array(common_gene_list)]
    except Exception as e:
        print(f"Error loading {celltype}: {e}")
        continue
# make a dataframe with the expression in each celltype for all genes
gene_annot_dict


In [ ]:
gene_annot_dict['cd14_mono']


In [ ]:
dfs = []
for celltype in gene_annot_dict.keys():
    df = gene_annot_dict[celltype].query('Chromosome == "chr10" | Chromosome == "chr11"')\
        .reset_index()[['obs', 'pred', 'gene_name']] \
        .groupby('gene_name').mean().reset_index()
    df['celltype'] = celltype
    dfs.append(df)
df = pd.concat(dfs)
df_pred = df.pivot(index='gene_name', columns='celltype', values='pred').dropna()
df_obs = df.pivot(index='gene_name', columns='celltype', values='obs').dropna()

# for each gene calculate the correlation between predicted and observed expression
corrs = []
for gene in df_pred.index:
    try:
        corr = df_pred.loc[gene].corr(df_obs.loc[gene])
        corrs.append((corr, df_pred.loc[gene].mean(), df_obs.loc[gene].mean(), gene))
    except Exception as e:
        print(f"Error calculating correlation for {gene}: {e}")
        continue
df_corr = pd.DataFrame(corrs, columns=['corr', 'pred', 'obs', 'gene_name'])



In [ ]:
# plot example genes
plt.rcParams['figure.dpi'] = 100

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11, 5))
df_corr['corr'].hist(bins=20, ax=ax[0])
# add total number of genes as title
ax[0].set_title(f'Total number of genes: {len(df_corr)}\nshared open genes across all celltypes \non leave out chr10 and chr11')
ax[0].set_xlabel('Cross-celltype Correlation')
ax[0].set_ylabel('Number of Genes')

sns.kdeplot(data=df_corr, x='pred', y='corr', ax=ax[1], shade=True)
ax[1].set_title('Predicted mean vs \nCross-celltype Correlation')
ax[1].set_xlabel('Predicted Expression')
ax[1].set_ylabel('Cross-celltype Correlation')
fig.tight_layout()


We can plot the predicted and observed expression to see whether there is any issue.

In [ ]:
df_pred